**`validate_occupancy`**

Scores curated `occupancy_type` against hand-labeled ground-truth points
and gates on per-class F1, so a change that improves one class by
wrecking another fails loudly.

Aggregate agreement is deliberately not the gate. It sat near 65% while
Single-Family accuracy was 37.6%, so a single scalar looked adequate
while the largest error mode in the dataset went unnoticed. Recall alone
is not the gate either: it would fail a change that trades a little
recall for more precision, and pass one that inflates recall by
labelling everything a single class.

# Configure

In [ ]:
import argparse
import sys
from pathlib import Path

import pandas as pd

# CHEER-specific configuration (counties, survey path, band collapse)
# lives in cheer_linkage.py beside this notebook; the linkage and scoring
# themselves are generic and live in openplaces.io.curator.validation.
# Resolve the repository root from any working directory (Jupyter runs
# notebooks from their own folder; scripts run from the repository root),
# so the import below works either way. It has to happen here rather than
# beside its first use, because the parser below defaults to a path this
# module defines.
root = Path.cwd()
while not (root / 'src' / 'openplaces').exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root / 'notebooks' / '05_curate'))
from openplaces.io.delivery import delivery_accuracy_dir  # noqa: E402
from openplaces.io.curator.validation import (  # noqa: E402
    compare_classifications_paired,
)
from cheer_linkage import (  # noqa: E402
    BASELINE_PATH,
    CLASSES,
    COUNTIES,
    align_to_baseline,
    load_baseline_predictions,
    save_baseline_predictions,
    check_baseline_coverage,
    link_ground_truth,
    score_sources,
)

In [ ]:
parser = argparse.ArgumentParser(description='Validate curated occupancy.')
parser.add_argument('--recipe_id', default='US_footprint-cheer-2026')
parser.add_argument('--admin_ids', nargs='*', default=None)
# Default None resolves to cheer_linkage.BASELINE_PATH below: the baseline
# is survey-derived (third-party data), so it lives in the cache tree,
# never beside the recipe in the repository.
parser.add_argument('--baseline', default=None)
parser.add_argument('--write_baseline', action='store_true')
# No tolerance any more. The old gate failed a class that lost more than
# 0.01 F1, which is inside this survey's own resampling spread (per-class
# sd 0.013-0.017 over 1,370 points), so it fired on noise and could not see
# a real regression smaller than about 0.03. The paired test below asks
# instead whether the change is distinguishable from zero at all.
parser.add_argument('--n_draws', type=int, default=400)
parser.add_argument('--seed', type=int, default=0)
# Scored tables go into the delivery's own `accuracies/` folder: how well
# the inventory scores travels with the inventory. Only the row-level
# linkage stays in the cache -- it carries survey addresses (see
# link_ground_truth's `save`), and the bundle is shared.
parser.add_argument(
    # The survey is NC-specific; the recipe ships two regions, and
    # delivery_accuracy_dir raises without a selector.
    '--out_dir',
    default=str(
        delivery_accuracy_dir('US_footprint-cheer-2026', region='cheer-eastern-nc')
    ),
)
parser.add_argument('--verbose', action='store_true')

# Test arguments

In [ ]:
ARGS_TEST = '--recipe_id US_footprint-cheer-2026 --verbose '

# Convert argument string to list of strings
args_list = [x for x in ARGS_TEST.split(' ') if x != '']

# Parse list of arguments
args = parser.parse_args(args_list)

# Display parsed arguments
args

# Validate occupancy against ground truth

In [ ]:
counties = tuple(args.admin_ids) if args.admin_ids else COUNTIES
linked = link_ground_truth(counties, verbose=args.verbose)
linked.shape

In [ ]:
# Score the vote and every input it arbitrates. A vote that scores worse
# than one of its own inputs on a class is discarding evidence, which a
# single-column score cannot show.
table = score_sources(linked)

out_dir = Path(args.out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
scores_path = out_dir / f'{args.recipe_id}_occupancy-scores.csv'
table.to_csv(scores_path, index=False)
print(f'wrote scores to {scores_path}')
table

In [ ]:
# Gate on a paired resample against the accepted run's own per-point
# predictions. Pairing is what makes this sensitive: the same survey points
# are drawn for both runs, so the sampling noise they share cancels and the
# interval reflects only the buildings whose class actually moved. An
# identical run therefore returns exactly [0, 0] rather than a spread.
baseline_path = args.baseline or BASELINE_PATH
if args.write_baseline:
    table.to_csv(baseline_path, index=False)
    written = save_baseline_predictions(linked)
    print(f'Baseline written: {baseline_path}')
    print(f'Baseline predictions written: {written}')
else:
    baseline = pd.read_csv(baseline_path)
    # A source missing from `table` would drop its baseline rows
    # from the merge silently, letting the gate pass on fewer
    # sources than it reports. Fail instead.
    check_baseline_coverage(table, baseline)

    predictions = load_baseline_predictions()
    truth, base_pred, new_pred, report = align_to_baseline(linked, predictions)
    print(
        f'paired on {report["n_shared"]:,} points '
        f'({report["n_baseline_only"]:,} baseline-only, '
        f'{report["n_current_only"]:,} current-only, '
        f'{report["n_truth_changed"]:,} hand labels changed)'
    )
    comparison = compare_classifications_paired(
        truth,
        base_pred,
        new_pred,
        list(CLASSES),
        n_draws=args.n_draws,
        seed=args.seed,
    )
    print(comparison.to_string(index=False))

    # Written before the gate raises, so a failing run leaves behind the
    # table that explains which class moved and by how much.
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    comparison_path = out_dir / f'{args.recipe_id}_occupancy-scores-vs-baseline.csv'
    comparison.to_csv(comparison_path, index=False)
    print()
    print(f'wrote comparison to {comparison_path}')

    # Fail only where the whole interval sits below zero -- that is a
    # regression the survey can actually resolve. A class that merely
    # drifted down within its own noise passes, which is the point.
    regressed = comparison[comparison['class'].ne('ALL') & (comparison['d_high'] < 0)]
    if len(regressed):
        classes = ', '.join(regressed['class'])
        raise SystemExit(f'FAIL: {len(regressed)} class(es) lost F1 ({classes})')

---
# Convert to script

*The above line and heading identify the end of the script.*

In [ ]:
from openplaces.flow import convert_to_script

try:
    convert_to_script(commit=True)
except Exception as error:
    # Headless execution (nbconvert) has no notebook context to resolve
    # the caller path from; run this cell interactively to commit the
    # script. Stripped from the converted script either way.
    print(f'convert_to_script skipped: {error}')